## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [16]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [17]:
MODEL = "openai/gpt-4.1-nano"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [18]:
# embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

import os
embeddings = OpenAIEmbeddings(
    model="openai/text-embedding-3-large",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv('OPENROUTER_API_KEY')
)
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [19]:
from dotenv import load_dotenv
load_dotenv(override=True)
import os

In [20]:
MODEL = "openai/gpt-4.1-nano"

In [21]:
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model=MODEL, 
    api_key=openrouter_api_key, base_url="https://openrouter.ai/api/v1")

### These LangChain objects implement the method `invoke()`

In [22]:
retriever.invoke("Who is Avery?")

[Document(id='b52da100-f0d9-4e74-a8dd-a438aa9e8956', metadata={'source': 'knowledge-base/employees/Avery Lancaster.md', 'doc_type': 'employees'}, page_content='# Avery Lancaster\n\n## Summary\n- **Date of Birth**: March 15, 1985\n- **Job Title**: Co-Founder & Chief Executive Officer (CEO)\n- **Location**: San Francisco, California\n- **Current Salary**: $225,000  \n\n## Insurellm Career Progression\n- **2015 - Present**: Co-Founder & CEO  \n  Avery Lancaster co-founded Insurellm in 2015 and has since guided the company to its current position as a leading Insurance Tech provider. Avery is known for her innovative leadership strategies and risk management expertise that have catapulted the company into the mainstream insurance market.  \n\n- **2013 - 2015**: Senior Product Manager at Innovate Insurance Solutions  \n  Before launching Insurellm, Avery was a leading Senior Product Manager at Innovate Insurance Solutions, where she developed groundbreaking insurance products aimed at the t

In [23]:
llm.invoke("Who is Avery?")

AIMessage(content="Could you please provide more context or specify which Avery you're referring to? There are many individuals and characters named Avery.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 11, 'total_tokens': 35, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'video_tokens': 0}, 'cost': 1.07e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': None, 'upstream_inference_prompt_cost': 1.1e-06, 'upstream_inference_completions_cost': 9.6e-06}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-4.1-nano', 'system_fingerprint': None, 'id': 'gen-1766939957-ch08zf6h3haIzrDVGuAM', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--cb6f35c9-7968-45a2-a70a-2e6b5041d4c0-0', usage_metadata={'input_tokens': 11, 'o

## Time to put this together!

In [24]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [25]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [26]:
answer_question("Who is Averi Lancaster?", [])

"It seems like there might be a typo in the name. Based on the provided information, you might be referring to Avery Lancaster. She is the Co-Founder and CEO of Insurellm, a leading insurance tech company, and has played a significant role in the company's growth and innovation since its founding in 2015. If you meant someone else or need more specific details, please let me know!"

## What could possibly come next? 😂

In [27]:
gr.ChatInterface(answer_question).launch()

/root/llm_engineering/.venv/lib/python3.12/site-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Admit it - you thought RAG would be more complicated than that!!